# Fiberwise clustering of $X(50,60)$

This notebook reproduces the manuscript's filament-clustering experiment with the current `circle_bundles` API. It writes a portable, pickle-free artifact for Notebook 03 while retaining the recovered notebook as provenance.

## Inputs and execution profile

The default is the manuscript-scale `paper` profile and the locally converted HC20 artifact. Override these with `OPTICAL_FLOW_PROFILE`, `OPTICAL_FLOW_PREPROCESSED`, and `OPTICAL_FLOW_BOUNDARY_OUTPUT`. The quick profile runs the clustering diagnostics but intentionally does not apply the manuscript-specific visual component identifications.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import circle_bundles as cb
import matplotlib.pyplot as plt
import numpy as np
from circle_bundles.analysis.fiberwise_clustering import (
    plot_fiberwise_pca_grid,
    plot_fiberwise_summary_bars,
)

from optical_flow_experiments import (
    assemble_paper_boundary_artifact,
    classify_component_h1,
    fit_fiberwise_clusters,
    load_experiment_config,
    load_preprocessed_npz,
    save_boundary_npz,
    select_dense_core,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "configs").is_dir() and (ROOT.parent / "configs").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Could not locate the repository root and its configs directory.")


def repo_relative_path(value, default):
    path = Path(value).expanduser() if value else Path(default)
    return path if path.is_absolute() else ROOT / path


profile_path = repo_relative_path(
    os.environ.get("OPTICAL_FLOW_PROFILE"),
    ROOT / "configs/paper.toml",
)
config = load_experiment_config(profile_path)
filament = config.filaments
print(f"Profile: {config.name}")
print(config.description)
print(f"Canonical random seed: {config.random_seed} (historical seed: {config.historical_seed})")

## Load $X(k,p)$

In [ ]:
artifact_override = os.environ.get("OPTICAL_FLOW_PREPROCESSED")
default_name = (
    "HC20_Flow_Patches.v1.npz" if config.name == "paper" else f"preprocessed_{config.name}.v1.npz"
)
artifact_path = repo_relative_path(
    artifact_override,
    ROOT / "data" / default_name,
)
if not artifact_path.is_file():
    raise RuntimeError(f"Preprocessed artifact not found: {artifact_path}")

patch_table = load_preprocessed_npz(artifact_path)
core = select_dense_core(
    patch_table,
    density_k=filament.density_k,
    density_fraction=filament.density_fraction,
)
data = core.data
assert len(data) == filament.expected_patch_count, (
    f"Expected {filament.expected_patch_count:,} points, observed {len(data):,}."
)
print(f"X({core.density_k}, {int(100 * core.density_fraction)}) contains {len(data):,} patches.")

In [ ]:
patch_vis = cb.make_patch_visualizer()
fig, axes = cb.show_data_vis(data, patch_vis, sampling_method=None, max_samples=28)
plt.show()

## Cluster fibers and filter the overlap graph

DBSCAN is run independently in 16 overlapping predominant-direction fibers. Local clusters are joined when they share samples, and edges of relative weight at most 0.07 are removed exactly as described in the manuscript.

In [ ]:
fit = fit_fiberwise_clusters(
    data,
    n_landmarks=filament.cover_landmarks,
    overlap=filament.cover_overlap,
    dbscan_epsilon=filament.dbscan_epsilon,
    dbscan_min_samples=filament.dbscan_min_samples,
    graph_weight_threshold=filament.graph_weight_threshold,
    build_pca_embeddings=True,
)
initial_ids = np.unique(fit.components[fit.components >= 0])
print(f"Initial global components: {len(initial_ids)}")
print(f"Initial unclustered points: {np.sum(fit.components == -1)}")
print(f"Largest initial component: {np.max(fit.summary['point_counts']):,} patches")
print(f"Removed graph edges: {fit.removed_edge_count}")
print(f"Filtered global components: {len(fit.component_memberships)}")

In [ ]:
fig, axes = plot_fiberwise_pca_grid(fit.summary, to_view=[3, 6, 14], n_cols=3)
plt.show()
fig, axes = plot_fiberwise_summary_bars(fit.summary, hide_biggest=True)
plt.show()

## Detect circular global components

For each filtered component, the longest $H_1$ interval is tested against the legacy sufficient condition $d > 2b$.

In [ ]:
h1 = classify_component_h1(
    data,
    fit.component_memberships,
    n_perm=500,
    random_state=config.random_seed,
)
circular_ids = np.flatnonzero(h1.circular)
print(f"Components with usable H1: {len(circular_ids)}")
print(circular_ids.tolist())

fig, ax = plt.subplots(figsize=(11, 4))
colors = np.where(h1.circular, "tab:blue", "lightgray")
ax.bar(np.arange(len(h1.lifetimes)), h1.lifetimes, color=colors)
ax.set(xlabel="Filtered component", ylabel="Longest H1 lifetime")
ax.grid(alpha=0.25)
plt.show()

## Construct the portable Notebook 03 input

The legacy analysis visually identified two incomplete circles as component pairs of recorded cardinalities `(1897, 1088)` and `(1612, 1120)`. Cardinalities replace execution-order-dependent component numbers here. The two circles receive deterministic synthetic bridge samples using seed 0.

In [ ]:
if config.name == "paper":
    boundary = assemble_paper_boundary_artifact(
        data,
        fit.component_memberships,
        h1.circular,
        random_seed=config.random_seed,
        synthetic_patches_per_fragment=filament.synthetic_patches_per_fragment,
    )
    boundary_override = os.environ.get("OPTICAL_FLOW_BOUNDARY_OUTPUT")
    boundary_path = repo_relative_path(
        boundary_override,
        ROOT / "data/K_50_60_Circles.v1.npz",
    )
    save_boundary_npz(boundary, boundary_path)
    print(f"Empirical filament patches: {boundary.metadata['empirical_patch_count']:,}")
    print(f"Synthetic bridge patches: {boundary.metadata['synthetic_patch_count']:,}")
    print(f"Augmented boundary sample: {len(boundary.patches):,}")
    print(f"Saved Notebook 03 input to {boundary_path}")
    historical_empirical_count = 55_001
    count_delta = boundary.metadata["empirical_patch_count"] - historical_empirical_count
    print(f"Difference from old recorded empirical count: {count_delta:+d} patches")
else:
    boundary = None
    print(
        "Quick profile complete; boundary assembly is defined only for the paper component structure."
    )

In [ ]:
if boundary is not None:
    representative_indices = [np.flatnonzero(row)[0] for row in boundary.memberships]
    representatives = boundary.patches[representative_indices]
    fig, axes = cb.show_data_vis(
        representatives,
        patch_vis,
        sampling_method=None,
        max_samples=28,
        n_cols=7,
    )
    plt.show()

## Record run metrics

In [ ]:
metrics = {
    "profile": config.name,
    "selected_patch_count": len(data),
    "initial_global_clusters": len(initial_ids),
    "initial_unclustered_points": int(np.sum(fit.components == -1)),
    "largest_initial_component_points": int(np.max(fit.summary["point_counts"])),
    "removed_edges": fit.removed_edge_count,
    "filtered_components": len(fit.component_memberships),
    "filtered_components_with_usable_h1": int(h1.circular.sum()),
    "boundary": None if boundary is None else boundary.metadata,
}
metrics_path = ROOT / "results" / config.name / "fiberwise_clustering_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n")
print(f"Wrote run metrics to {metrics_path}")
metrics